# P1 폐렴 3분류 — 학습 v1 (베이스라인, 원본 val 사용)

**v1 설계 포인트:**
- 데이터 분할: **원본 train/val/test 그대로** (val은 제공된 16장 그대로 사용)
  - ⚠️ 원본 val은 16장이고 바이러스성 0개 → 검증이 부실함. 이 한계를 직접 확인하는 게 v1 목적.
  - (v2에서 train 환자단위 재분할로 개선 예정)
- 모델: ImageNet 사전학습 ResNet18 → 3클래스
- 입력 224, 가벼운 증강, 클래스 가중치 손실
- 결과를 **S3에 run 단위로 정리 저장** (config + metrics + 모델)

### ⚠️ 실행 전
1. GPU 인스턴스(`ml.g4dn.xlarge`)로 전환 → CELL 0에서 `CUDA True`
2. `pneumonia_labels.csv` 업로드
3. 끝나면 인스턴스 **Stop**

## CELL 0 — 디바이스 확인

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| CUDA:', torch.cuda.is_available())
if device.type != 'cuda':
    print('⚠️ GPU 아님. ml.g4dn.xlarge 로 바꾸면 빠름 (CPU로도 되지만 느림)')

## CELL 1 — 설정 + 데이터 분할 (v1: 원본 val 그대로)
`VERSION`, 하이퍼파라미터는 여기서 바꾼다. 바꿔서 다시 돌리면 S3에 새 run 폴더로 저장됨.

In [ ]:
import io, json, time, numpy as np, pandas as pd, boto3
from PIL import Image

# ===== 버전/실험 식별 =====
VERSION = 'v1'                 # 노트북 버전 (코드 구조 단위)
NOTE = 'baseline_original_val' # 이번 run 설명 (하이퍼파라미터 바꾸면 메모 변경)

# ===== 설정 =====
DATA_BUCKET = 'aws-say3-dataset1'          # 읽기용 (이미지)
OUT_BUCKET  = 'aws-say3-student-dataset'    # 쓰기용 (결과 저장)
S3_PREFIX = 'pneumonia-xray/chest_xray/'   # 이미지 경로 접두사
S3_DEV_ROOT = 'students/aws-say3-11/P1/model_dev'   # 결과 저장 루트 (쓰기 버킷 안)
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
NUM_CLASSES = 3
CLASS_NAMES = ['NORMAL', 'BACTERIAL_PNEUMONIA', 'VIRAL_PNEUMONIA']

# ----- 하이퍼파라미터 (바꿔가며 실험하는 부분) -----
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-4
SEED = 42

torch.manual_seed(SEED); np.random.seed(SEED)
s3 = boto3.client('s3')

# ===== 데이터 분할: v1은 원본 그대로 =====
df = pd.read_csv('pneumonia_labels.csv')
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

print('train', len(train_df), '/ val', len(val_df), '/ test', len(test_df))
print('train 분포:', train_df['label'].value_counts().sort_index().to_dict())
print('val 분포:  ', val_df['label'].value_counts().sort_index().to_dict())
print('⚠️ val 분포 확인 — 바이러스성(2)이 0개면 그 클래스는 val에서 평가 불가')

## CELL 2 — Dataset / DataLoader (S3에서 이미지 읽기)

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(7),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(), T.Normalize(MEAN, STD),
])

class S3DS(Dataset):
    def __init__(self, rows, tf):
        self.rows = rows.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        o = s3.get_object(Bucket=DATA_BUCKET, Key=S3_PREFIX + r['relative_path'])
        im = Image.open(io.BytesIO(o['Body'].read())).convert('RGB')
        return self.tf(im), int(r['label'])

train_loader = DataLoader(S3DS(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(S3DS(val_df, eval_tf),    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print('배치 수 — train', len(train_loader), '/ val', len(val_loader))

## CELL 3 — 모델 / 손실 / 최적화

In [ ]:
import torch.nn as nn
import torchvision.models as models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

counts = train_df['label'].value_counts().sort_index()
weights = (counts.sum() / (NUM_CLASSES * counts)).values
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
print('클래스 가중치:', class_weights.cpu().numpy().round(3).tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

## CELL 4 — 학습 루프 (에폭별 상세 로그 + best 저장)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def run_epoch(loader, train):
    model.train() if train else model.eval()
    tot, preds, labs = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs); loss = criterion(out, labels)
            if train: loss.backward(); optimizer.step()
            tot += loss.item() * imgs.size(0)
            preds += out.argmax(1).cpu().tolist(); labs += labels.cpu().tolist()
    return tot/len(loader.dataset), accuracy_score(labs, preds), f1_score(labs, preds, average='macro')

history = []
best_f1 = -1
print(f'학습 시작 — {EPOCHS} epochs, batch {BATCH_SIZE}, lr {LR}\n' + '='*60)
run_start = time.time()
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, True)
    va_loss, va_acc, va_f1 = run_epoch(val_loader, False)
    dt = time.time() - t0
    mark = ''
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save({'model_state': model.state_dict(), 'class_names': CLASS_NAMES,
                    'img_size': IMG_SIZE, 'mean': MEAN, 'std': STD}, 'best_model.pth')
        mark = '  ← best 갱신 ✅ (저장)'
    print(f'[Epoch {epoch:02d}/{EPOCHS}] '
          f'train: loss {tr_loss:.3f} acc {tr_acc:.3f} f1 {tr_f1:.3f}  |  '
          f'val: loss {va_loss:.3f} acc {va_acc:.3f} f1 {va_f1:.3f}  '
          f'({dt:.0f}s){mark}')
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc, 'train_f1': tr_f1,
                    'val_loss': va_loss, 'val_acc': va_acc, 'val_f1': va_f1, 'sec': round(dt,1)})
total_sec = time.time() - run_start
print('='*60)
print(f'학습 완료. best val macro-F1 = {best_f1:.4f}  |  총 {total_sec/60:.1f}분')

## CELL 5 — test 평가 (F1 / 혼동행렬)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

ckpt = torch.load('best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state']); model.eval()

test_loader = DataLoader(S3DS(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
preds, labs = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds += model(imgs.to(device)).argmax(1).cpu().tolist(); labs += labels.tolist()

test_acc = accuracy_score(labs, preds)
test_f1 = f1_score(labs, preds, average='macro')
print(f'[Test] 정확도 {test_acc:.4f} | macro-F1 {test_f1:.4f}\n')
report = classification_report(labs, preds, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(labs, preds, target_names=CLASS_NAMES, digits=4))
cm = confusion_matrix(labs, preds)
print('혼동행렬 (행=실제, 열=예측):')
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

## CELL 6 — S3에 run 단위로 정리 저장
구조: `s3://{OUT_BUCKET}/{S3_DEV_ROOT}/{VERSION}/run_{시각}_{NOTE}/`
각 run에 **모델 + config + metrics** 를 함께 저장 → 하이퍼파라미터만 바꿔 돌려도 폴더가 갈려 둘 다 보존.

In [ ]:
run_id = time.strftime('%Y%m%d_%H%M')
run_name = f'run_{run_id}_{NOTE}'
prefix = f'{S3_DEV_ROOT}/{VERSION}/{run_name}'

# 1) 설정(config)
config_dict = {
    'version': VERSION, 'note': NOTE, 'run_id': run_id,
    'split': 'original (train/val/test 그대로)',
    'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
    'lr': LR, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'model': 'resnet18 (ImageNet pretrained)',
    'augment': 'HFlip0.5 + Rotate7 + ColorJitter0.1',
    'class_weights': class_weights.cpu().numpy().round(4).tolist(),
}
# 2) 성능(metrics)
metrics_dict = {
    'best_val_f1': round(best_f1, 4),
    'test_acc': round(test_acc, 4),
    'test_macro_f1': round(test_f1, 4),
    'test_report': report,
    'confusion_matrix': cm.tolist(),
    'history': history,
}

with open('config.json', 'w') as f: json.dump(config_dict, f, ensure_ascii=False, indent=2)
with open('metrics.json', 'w') as f: json.dump(metrics_dict, f, ensure_ascii=False, indent=2)

# 3) S3 업로드 (모델 + config + metrics)
for fname in ['best_model.pth', 'config.json', 'metrics.json']:
    s3.upload_file(fname, OUT_BUCKET, f'{prefix}/{fname}')
    print('업로드:', f's3://{OUT_BUCKET}/{prefix}/{fname}')

print(f'\n[OK] 이 run 저장 완료 -> s3://{OUT_BUCKET}/{prefix}/')

# ============================================================
# 4) 로컬 다운로드 명령어 생성
#    아래 출력된 명령을 복사해 "맥 로컬 터미널"에 붙여넣으면
#    이번 run 결과만 로컬 정리 위치로 다운로드된다. (json만, 가벼움)
#    모델 .pth 까지 받으려면 명령 끝의  --exclude "*.pth"  를 지운다.
# ============================================================
LOCAL_DEST = '/Users/skku_aws3_16/Documents/안재형/PROJECT/train/model_dev/_s3_results'
PROFILE = 'aws-say3-11'
local_run_dir = f'{LOCAL_DEST}/{VERSION}/{run_name}'
s3_run_uri = f's3://{OUT_BUCKET}/{prefix}/'

dl_cmd = (
    f'mkdir -p "{local_run_dir}" && '
    f'aws s3 cp "{s3_run_uri}" "{local_run_dir}" --recursive '
    f'--profile {PROFILE} --exclude "*.pth"'
)

print('\n' + '=' * 72)
print('아래 명령을 복사해서 [맥 로컬 터미널]에 붙여넣으세요 (이 run만 다운로드):')
print('  * json만 받음. 모델까지 받으려면 끝의  --exclude "*.pth"  삭제')
print('=' * 72)
print(dl_cmd)
print('=' * 72)
